In [1]:
!pip install nbformat

In [3]:
import numpy as np
import pandas as pd
df = pd.DataFrame([
    [8, 6, 1],
    [9, 7, 1],
    [7, 5, 1],
    [3, 2, 0],
    [2, 1, 0],
    [4, 3, 0]
], columns=['cgpa', 'study_hours', 'result'])

print(df)

   cgpa  study_hours  result
0     8            6       1
1     9            7       1
2     7            5       1
3     3            2       0
4     2            1       0
5     4            3       0


In [4]:
# Sigmoid squishes ANY number between 0 and 1
# This gives us PROBABILITY
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


In [5]:
print(sigmoid(-10))   # → 0.00004  (almost 0 = FAIL)
print(sigmoid(0))     # → 0.5      (50/50)
print(sigmoid(10))    # → 0.99999  (almost 1 = PASS)


4.5397868702434395e-05
0.5
0.9999546021312976


In [ ]:

# Rule:
# output > 0.5 → predict 1 (PASS)
# output < 0.5 → predict 0 (FAIL)
```
```
Sigmoid Graph:

1.0  |          .........
0.8  |       ...
0.5  |     .        ← decision boundary
0.2  |  ...
0.0  |...
      ──────────────────
     -10   -5   0   5   1

In [6]:
np.random.seed(42)

# Layer 1 weights (2 inputs → 2 hidden neurons)
W1 = np.ones((2, 2)) * 0.1
b1 = np.zeros((2, 1))

# Layer 2 weights (2 hidden → 1 output)
W2 = np.ones((2, 1)) * 0.1
b2 = np.zeros((1, 1))

print("W1:\n", W1)
print("W2:\n", W2)

W1:
 [[0.1 0.1]
 [0.1 0.1]]
W2:
 [[0.1]
 [0.1]]


In [7]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    return sigmoid(z) * (1 - sigmoid(z))
    # needed for backpropagation later

def predict(X, W1, b1, W2, b2):
    # Layer 1: input → hidden
    Z1     = np.dot(W1.T, X) + b1     # raw value
    A1     = sigmoid(Z1)               # apply sigmoid ← NEW!

    # Layer 2: hidden → output
    Z2     = np.dot(W2.T, A1) + b2    # raw value
    A2     = sigmoid(Z2)               # apply sigmoid ← NEW!

    # A2 is now a PROBABILITY between 0 and 1
    return A2, A1, Z1, Z2

# Test with student 1 (cgpa=8, study=6, actual=PASS=1)
X = np.array([[8],
              [6]])
y_actual = 1

A2, A1, Z1, Z2 = predict(X, W1, b1, W2, b2)

print("Hidden layer output A1:\n", A1)
print("Final prediction A2   :", A2[0][0])
print("Probability of PASS   :", round(A2[0][0]*100, 2), "%")
print("Predicted class       :", 1 if A2[0][0] > 0.5 else 0)

Hidden layer output A1:
 [[0.80218389]
 [0.80218389]]
Final prediction A2   : 0.5400233812168658
Probability of PASS   : 54.0 %
Predicted class       : 1


In [8]:
# For classification we use Binary Cross Entropy
# NOT MSE like in regression
def binary_cross_entropy(y_actual, y_pred):
    # small value added to avoid log(0) error
    epsilon = 1e-8
    loss = -(y_actual * np.log(y_pred + epsilon) +
            (1 - y_actual) * np.log(1 - y_pred + epsilon))
    return loss

# Example:
# actual=1, pred=0.9  → loss = small  ✅ good prediction
# actual=1, pred=0.1  → loss = large  ❌ bad prediction

loss = binary_cross_entropy(y_actual, A2[0][0])
print("Loss:", loss)

# Formula:
# Loss = -[y*log(p) + (1-y)*log(1-p)]
# y=1, p=0.9 → Loss = -[1*log(0.9)] = 0.10  small ✅
# y=1, p=0.1 → Loss = -[1*log(0.1)] = 2.30  large ❌

Loss: 0.6161428232937041


In [10]:
learning_rate = 0.01   # slightly higher than regression

def update_weights(X, y_actual, A2, A1, Z1, Z2,
                   W1, b1, W2, b2):

    # ── Output layer error ────────────────────────────
    # dL/dZ2 = A2 - y  (derivative of BCE + sigmoid)
    dZ2 = A2 - y_actual          # output error

    # ── Update W2 and b2 ──────────────────────────────
    dW2 = np.dot(A1, dZ2.T)      # how much to change W2
    db2 = dZ2                    # how much to change b2

    W2  = W2 - learning_rate * dW2
    b2  = b2 - learning_rate * db2

    # ── Hidden layer error ────────────────────────────
    # error travels back through W2
    dZ1 = np.dot(W2, dZ2) * sigmoid_derivative(Z1)

    # ── Update W1 and b1 ──────────────────────────────
    dW1 = np.dot(X, dZ1.T)
    db1 = dZ1

    W1  = W1 - learning_rate * dW1
    b1  = b1 - learning_rate * db1

    return W1, b1, W2, b2

# NOTE: we use MINUS now (not plus like regression)
# because BCE derivative works differently

In [11]:
# Reset weights
W1 = np.ones((2, 2)) * 0.1
b1 = np.zeros((2, 1))
W2 = np.ones((2, 1)) * 0.1
b2 = np.zeros((1, 1))

epochs = 1000

for epoch in range(epochs):

    total_loss = 0

    for i in range(len(df)):

        # Get student data
        X = np.array([[df['cgpa'][i]],
                      [df['study_hours'][i]]])
        y_actual = df['result'][i]

        # Step 1: Predict
        A2, A1, Z1, Z2 = predict(X, W1, b1, W2, b2)

        # Step 2: Calculate loss
        loss = binary_cross_entropy(y_actual, A2[0][0])
        total_loss += loss

        # Step 3: Fix weights
        W1, b1, W2, b2 = update_weights(
            X, y_actual, A2, A1, Z1, Z2,
            W1, b1, W2, b2
        )

    # Print every 100 epochs
    if (epoch + 1) % 100 == 0:
        avg_loss = total_loss / len(df)
        print(f"Epoch {epoch+1:4d} | Loss: {avg_loss:.4f}")

Epoch  100 | Loss: 0.6738
Epoch  200 | Loss: 0.6570
Epoch  300 | Loss: 0.6373
Epoch  400 | Loss: 0.6131
Epoch  500 | Loss: 0.5830
Epoch  600 | Loss: 0.5456
Epoch  700 | Loss: 0.5010
Epoch  800 | Loss: 0.4510
Epoch  900 | Loss: 0.3990
Epoch 1000 | Loss: 0.3489


In [12]:
print("\n--- Final Results ---")
print(f"{'CGPA':<6} {'Study':<8} {'Actual':<10} {'Prob%':<10} {'Predicted'}")
print("-" * 45)

correct = 0

for i in range(len(df)):
    X = np.array([[df['cgpa'][i]],
                  [df['study_hours'][i]]])
    y_actual = df['result'][i]

    A2, _, _, _ = predict(X, W1, b1, W2, b2)

    probability  = A2[0][0]
    y_pred_class = 1 if probability > 0.5 else 0
    label        = "PASS" if y_pred_class == 1 else "FAIL"

    if y_pred_class == y_actual:
        correct += 1

    print(f"{df['cgpa'][i]:<6} "
          f"{df['study_hours'][i]:<8} "
          f"{'PASS' if y_actual==1 else 'FAIL':<10} "
          f"{probability*100:<10.1f} "
          f"{label}")

print(f"\nAccuracy: {correct}/{len(df)} = "
      f"{correct/len(df)*100:.1f}%")



--- Final Results ---
CGPA   Study    Actual     Prob%      Predicted
---------------------------------------------
8      6        PASS       75.0       PASS
9      7        PASS       79.1       PASS
7      5        PASS       67.9       PASS
3      2        FAIL       30.0       FAIL
2      1        FAIL       20.8       FAIL
4      3        FAIL       43.0       FAIL

Accuracy: 6/6 = 100.0%


In [ ]:
## 🔑 KEY FORMULAS
```
Sigmoid          →   σ(z) = 1 / (1 + e^(-z))

Forward Pass     →   Z1 = W1ᵀ·X + b1  →  A1 = σ(Z1)
                     Z2 = W2ᵀ·A1 + b2  →  A2 = σ(Z2)

BCE Loss         →   L = -[y·log(p) + (1-y)·log(1-p)]

Backprop         →   dZ2 = A2 - y
                     W2  = W2 - α·(A1·dZ2ᵀ)
                     dZ1 = W2·dZ2 × σ'(Z1)
                     W1  = W1 - α·(X·dZ1ᵀ)

Decision Rule    →   prob > 0.5 → Class 1 (PASS)
                     prob < 0.5 → Class 0 (FAIL)